# SNA Feature Engineering (MVP) for Fraud Detection

本 notebook 實作 `strategy/SNA_feature.md` 的 **MVP 版本**，目標是：

1. 建立「只看過去」的 SNA 特徵（避免資料洩漏）
2. 與 baseline features 結合
3. 用 LightGBM / XGBoost（若環境可用）做快速對照

> 這版先做 MVP 8 個核心 SNA 特徵 + 一個簡單對照流程。


In [ ]:
# ==== 0) Imports ====
import warnings
warnings.filterwarnings('ignore')

from collections import defaultdict

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve

# 可選：LightGBM / XGBoost
HAS_LGB = True
HAS_XGB = True
try:
    import lightgbm as lgb
except Exception:
    HAS_LGB = False

try:
    from xgboost import XGBClassifier
except Exception:
    HAS_XGB = False

import matplotlib.pyplot as plt


In [ ]:
# ==== 1) Config ====
DATA_PATH = '../Transactions Data.csv'

# 開發模式建議先抽樣，筆電較穩定
USE_SAMPLE = True
SAMPLE_FRAC = 0.20
RANDOM_STATE = 42

# 時間切分比例
TRAIN_RATIO = 0.70
VALID_RATIO = 0.15
TEST_RATIO = 0.15

# 是否先限制資料行數（debug用）
MAX_ROWS = None  # e.g. 500000


In [ ]:
# ==== 2) Load data ====
df = pd.read_csv(DATA_PATH)

if MAX_ROWS is not None:
    df = df.iloc[:MAX_ROWS].copy()

# 保持時間順序
if USE_SAMPLE:
    # 先依 step stratified-like sampling：每個 step 抽固定比例
    df = (
        df.groupby('step', group_keys=False)
          .apply(lambda g: g.sample(frac=min(SAMPLE_FRAC, 1.0), random_state=RANDOM_STATE) if len(g) > 1 else g)
          .reset_index(drop=True)
    )

# 關鍵：排序後做「只看過去」特徵
df = df.sort_values('step').reset_index(drop=True)

print('Shape:', df.shape)
print('Fraud rate:', df['isFraud'].mean())
print(df.head(3))


## 3) MVP SNA Features（Incremental, Leakage-safe）

以下 8 個特徵依序逐筆更新：

1. `orig_out_degree_hist`
2. `dest_in_degree_hist`
3. `orig_unique_dest_hist`
4. `dest_unique_orig_hist`
5. `pair_txn_count_hist`
6. `is_first_time_pair`
7. `orig_out_amount_sum_hist`
8. `pair_avg_amount_hist`

每筆交易的特徵都只用到「之前」資料，然後才更新統計。


In [ ]:
# ==== 3) Build leakage-safe MVP SNA features ====
work = df.copy()

# counters/states
orig_out_degree = defaultdict(int)
dest_in_degree = defaultdict(int)
orig_out_amount_sum = defaultdict(float)

pair_count = defaultdict(int)
pair_amount_sum = defaultdict(float)

orig_to_dests = defaultdict(set)
dest_from_origs = defaultdict(set)

# feature arrays
n = len(work)
feat_orig_out_degree_hist = np.zeros(n, dtype=np.int32)
feat_dest_in_degree_hist = np.zeros(n, dtype=np.int32)
feat_orig_unique_dest_hist = np.zeros(n, dtype=np.int32)
feat_dest_unique_orig_hist = np.zeros(n, dtype=np.int32)
feat_pair_txn_count_hist = np.zeros(n, dtype=np.int32)
feat_is_first_time_pair = np.zeros(n, dtype=np.int8)
feat_orig_out_amount_sum_hist = np.zeros(n, dtype=np.float64)
feat_pair_avg_amount_hist = np.zeros(n, dtype=np.float64)

orig_col = work['nameOrig'].values
dest_col = work['nameDest'].values
amt_col = work['amount'].values

for i in range(n):
    o = orig_col[i]
    d = dest_col[i]
    a = float(amt_col[i])
    pair = (o, d)

    # ----- read history (before current txn) -----
    c_pair = pair_count[pair]

    feat_orig_out_degree_hist[i] = orig_out_degree[o]
    feat_dest_in_degree_hist[i] = dest_in_degree[d]
    feat_orig_unique_dest_hist[i] = len(orig_to_dests[o])
    feat_dest_unique_orig_hist[i] = len(dest_from_origs[d])
    feat_pair_txn_count_hist[i] = c_pair
    feat_is_first_time_pair[i] = 1 if c_pair == 0 else 0
    feat_orig_out_amount_sum_hist[i] = orig_out_amount_sum[o]
    feat_pair_avg_amount_hist[i] = (pair_amount_sum[pair] / c_pair) if c_pair > 0 else 0.0

    # ----- update states with current txn -----
    orig_out_degree[o] += 1
    dest_in_degree[d] += 1
    orig_out_amount_sum[o] += a

    pair_count[pair] += 1
    pair_amount_sum[pair] += a

    orig_to_dests[o].add(d)
    dest_from_origs[d].add(o)

work['orig_out_degree_hist'] = feat_orig_out_degree_hist
work['dest_in_degree_hist'] = feat_dest_in_degree_hist
work['orig_unique_dest_hist'] = feat_orig_unique_dest_hist
work['dest_unique_orig_hist'] = feat_dest_unique_orig_hist
work['pair_txn_count_hist'] = feat_pair_txn_count_hist
work['is_first_time_pair'] = feat_is_first_time_pair
work['orig_out_amount_sum_hist'] = feat_orig_out_amount_sum_hist
work['pair_avg_amount_hist'] = feat_pair_avg_amount_hist

print('SNA MVP features added.')
work[['orig_out_degree_hist','dest_in_degree_hist','pair_txn_count_hist','is_first_time_pair']].head()


In [ ]:
# ==== 4) Add existing baseline engineered features ====
work['deltaOrig'] = work['oldbalanceOrg'] - work['newbalanceOrig']
work['deltaDest'] = work['newbalanceDest'] - work['oldbalanceDest']
work['isOrigBalanceZero'] = (work['oldbalanceOrg'] == 0).astype(int)
work['isDestBalanceZero'] = (work['oldbalanceDest'] == 0).astype(int)

TARGET = 'isFraud'


In [ ]:
# ==== 5) Time-based split ====
steps_sorted = np.sort(work['step'].unique())
train_cut_idx = int(len(steps_sorted) * TRAIN_RATIO)
valid_cut_idx = int(len(steps_sorted) * (TRAIN_RATIO + VALID_RATIO))

train_max_step = steps_sorted[train_cut_idx - 1]
valid_max_step = steps_sorted[valid_cut_idx - 1]

train_mask = work['step'] <= train_max_step
valid_mask = (work['step'] > train_max_step) & (work['step'] <= valid_max_step)
test_mask = work['step'] > valid_max_step

print('Train size:', train_mask.sum(), 'fraud rate:', work.loc[train_mask, TARGET].mean())
print('Valid size:', valid_mask.sum(), 'fraud rate:', work.loc[valid_mask, TARGET].mean())
print('Test  size:', test_mask.sum(),  'fraud rate:', work.loc[test_mask, TARGET].mean())


## 6) Base vs Base+SNA 設計

- **Base**：原先 baseline 欄位（不含 ID）
- **Base+SNA**：Base + 8 個 MVP SNA 特徵


In [ ]:
# ==== 6) Feature sets ====
base_num = [
    'step', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud',
    'deltaOrig', 'deltaDest', 'isOrigBalanceZero', 'isDestBalanceZero'
]
base_cat = ['type']

sna_num = [
    'orig_out_degree_hist',
    'dest_in_degree_hist',
    'orig_unique_dest_hist',
    'dest_unique_orig_hist',
    'pair_txn_count_hist',
    'is_first_time_pair',
    'orig_out_amount_sum_hist',
    'pair_avg_amount_hist'
]

base_features = base_num + base_cat
base_sna_features = base_num + base_cat + sna_num

# 不直接使用 nameOrig/nameDest

X_base = work[base_features].copy()
X_base_sna = work[base_sna_features].copy()
y = work[TARGET].values

y_train = y[train_mask]
y_valid = y[valid_mask]
y_test = y[test_mask]


In [ ]:
# ==== 7) Helpers ====
def evaluate_probs(y_true, y_prob, name='model'):
    ap = average_precision_score(y_true, y_prob)
    roc = roc_auc_score(y_true, y_prob)
    print(f'[{name}] PR-AUC={ap:.6f} | ROC-AUC={roc:.6f}')
    return {'model': name, 'pr_auc': ap, 'roc_auc': roc}


def plot_pr(y_true, y_prob, title='PR curve'):
    p, r, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    plt.figure(figsize=(6,4))
    plt.plot(r, p, label=f'AP={ap:.4f}')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(title)
    plt.grid(alpha=0.2)
    plt.legend()
    plt.show()


In [ ]:
# ==== 8) Logistic: Base vs Base+SNA ====
# Base preprocess
pre_base = ColumnTransformer([
    ('num', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), base_num),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore'))
    ]), base_cat)
])

# Base+SNA preprocess
pre_base_sna = ColumnTransformer([
    ('num', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), base_num + sna_num),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore'))
    ]), base_cat)
])

logit_base = Pipeline([
    ('prep', pre_base),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', solver='saga', n_jobs=-1, random_state=RANDOM_STATE))
])

logit_base_sna = Pipeline([
    ('prep', pre_base_sna),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', solver='saga', n_jobs=-1, random_state=RANDOM_STATE))
])

Xb_train, Xb_valid = X_base.loc[train_mask], X_base.loc[valid_mask]
Xbs_train, Xbs_valid = X_base_sna.loc[train_mask], X_base_sna.loc[valid_mask]

logit_base.fit(Xb_train, y_train)
logit_base_sna.fit(Xbs_train, y_train)

p_base = logit_base.predict_proba(Xb_valid)[:, 1]
p_base_sna = logit_base_sna.predict_proba(Xbs_valid)[:, 1]

res = []
res.append(evaluate_probs(y_valid, p_base, 'Logit-Base (Valid)'))
res.append(evaluate_probs(y_valid, p_base_sna, 'Logit-Base+SNA (Valid)'))

plot_pr(y_valid, p_base, 'Logit Base - Valid PR')
plot_pr(y_valid, p_base_sna, 'Logit Base+SNA - Valid PR')

pd.DataFrame(res)


In [ ]:
# ==== 9) LightGBM (optional): Base+SNA quick run ====
if HAS_LGB:
    lgb_features = base_num + sna_num + ['type']
    lgb_df = work[lgb_features + [TARGET]].copy()

    # LightGBM 類別欄位可直接 category
    lgb_df['type'] = lgb_df['type'].astype('category')

    train_df = lgb_df.loc[train_mask]
    valid_df = lgb_df.loc[valid_mask]

    X_train_lgb = train_df.drop(columns=[TARGET])
    y_train_lgb = train_df[TARGET]
    X_valid_lgb = valid_df.drop(columns=[TARGET])
    y_valid_lgb = valid_df[TARGET]

    neg = (y_train_lgb == 0).sum()
    pos = (y_train_lgb == 1).sum()
    spw = max(1.0, neg / max(pos, 1))

    model_lgb = lgb.LGBMClassifier(
        objective='binary',
        n_estimators=400,
        learning_rate=0.05,
        num_leaves=64,
        min_child_samples=100,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=spw,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model_lgb.fit(
        X_train_lgb, y_train_lgb,
        eval_set=[(X_valid_lgb, y_valid_lgb)],
        eval_metric='average_precision',
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    p_lgb = model_lgb.predict_proba(X_valid_lgb)[:, 1]
    _ = evaluate_probs(y_valid_lgb, p_lgb, 'LightGBM-Base+SNA (Valid)')
    plot_pr(y_valid_lgb, p_lgb, 'LightGBM Base+SNA - Valid PR')
else:
    print('LightGBM not installed; skip this block.')


In [ ]:
# ==== 10) XGBoost (optional): Base+SNA quick run ====
if HAS_XGB:
    xgb_df = work[base_num + base_cat + sna_num + [TARGET]].copy()

    # one-hot for xgboost quick baseline
    xgb_df = pd.get_dummies(xgb_df, columns=['type'], drop_first=False)

    train_df = xgb_df.loc[train_mask]
    valid_df = xgb_df.loc[valid_mask]

    X_train_xgb = train_df.drop(columns=[TARGET])
    y_train_xgb = train_df[TARGET]
    X_valid_xgb = valid_df.drop(columns=[TARGET])
    y_valid_xgb = valid_df[TARGET]

    neg = (y_train_xgb == 0).sum()
    pos = (y_train_xgb == 1).sum()
    spw = max(1.0, neg / max(pos, 1))

    model_xgb = XGBClassifier(
        objective='binary:logistic',
        eval_metric='aucpr',
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=spw,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method='hist'
    )

    model_xgb.fit(
        X_train_xgb, y_train_xgb,
        eval_set=[(X_valid_xgb, y_valid_xgb)],
        verbose=False
    )

    p_xgb = model_xgb.predict_proba(X_valid_xgb)[:, 1]
    _ = evaluate_probs(y_valid_xgb, p_xgb, 'XGBoost-Base+SNA (Valid)')
    plot_pr(y_valid_xgb, p_xgb, 'XGBoost Base+SNA - Valid PR')
else:
    print('XGBoost not installed; skip this block.')


## 11) 下一步

1. 先確認 **Logit Base vs Base+SNA** 是否有 PR-AUC 提升。  
2. 若提升成立，再在 LightGBM/XGBoost 上做 threshold + risk tier。  
3. 若資源吃緊，優先保留貢獻較高的 SNA 特徵，移除低價值特徵。  
4. 最後固定流程到 Test 集做最終報告。
